In [ ]:
#@title Cell 52.1 — Purpose and fixed Figure 3 design
# Figure 3: BMC-N predictive MIC intervals for individual resistant E. coli pathogens.
# Panel A: cefepime; Panel B: gentamicin.
# Resistance is determined ONLY by observed MIC (EUCAST 2026):
#    cefepime: observed MIC > 4 mg/L
#    gentamicin: observed MIC > 2 mg/L
# From each antibiotic's resistant observations, select 15 deterministically:
# 5 from the lower third, 5 from the middle third, 5 from the upper third
# of the absolute original Model C prediction error distribution.
# No selection based on interval coverage, and no recalibration or re-fitting.
# Original Model C point prediction remains unchanged.
# 95% predictive MIC interval comes directly from the validated NB41 BMC-N output.

print('NB52: Figure 3 — BMC-N intervals for cefepime- and gentamicin-resistant E. coli.')
print('Input: validated Notebook 41 held-out BMC-N prediction table ONLY.')
print('Outputs: selected-pathogen CSV, full resistant-cohort audit, Figure 3 PNG and PDF.')

In [ ]:
#@title Cell 52.2 — Locate the existing NB41 output and define NB52 results folder
import os
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass  # Also works outside Colab when BMC_PROJECT_ROOT is configured.

ROOT = Path(os.environ.get('BMC_PROJECT_ROOT', '/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(
        f'RefMIC project folder not found: {ROOT}. '
        'If using a different location, set BMC_PROJECT_ROOT.'
    )

INPUT_FOLDER = '41_BMC_N_Pathogen_Out_Validation'
INPUT_NAME = '41_BMC_N_independent_outer_test_predictions.csv.gz'
EXPECTED = ROOT / '03_Results' / INPUT_FOLDER / INPUT_NAME
OUT = ROOT / '03_Results' / '52_BMC_Figure3_Resistant_Pathogens'
OUT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for part in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(part)
    return h.hexdigest()

def find_nb41_output():
    if EXPECTED.is_file():
        return EXPECTED
    search_root = Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
    matches = []
    for dirname, dirs, filenames in os.walk(search_root):
        dirs[:] = [d for d in dirs if d not in
                   {'.git', '.Trashes', '.Trash-1000', '__pycache__', 'node_modules'}]
        if INPUT_NAME in filenames:
            matches.append(Path(dirname) / INPUT_NAME)
    if not matches:
        raise FileNotFoundError(
            f'NB41 output not found: {INPUT_NAME}\n'
            f'Expected location: {EXPECTED}\n'
            'Run Notebook 41 Cell 41.8 to save its validated predictions, '
            'then rerun this cell. NB26 is not required.'
        )
    if len(matches) > 1 and len({sha256_file(p) for p in matches}) > 1:
        raise ValueError(
            'Different copies of the NB41 output were found.\n'
            'Please place the authoritative file at the expected location:\n'
            f'{EXPECTED}\nFound:\n' + '\n'.join(map(str, matches))
        )
    return sorted(matches, key=lambda p: (0 if ROOT in p.parents else 1,
                                          len(p.parts), str(p)))[0]

BMCN_PATH = find_nb41_output()
print('BMC-N held-out file:', BMCN_PATH)
print('Figure 3 results folder:', OUT)

RESISTANCE_LIMIT_MG_L = {'cefepime': 4.0, 'gentamicin': 2.0}

# The complete input audit is reusable. Cell 52.4 can restore the cohort
# if Cell 52.3 was skipped or its variables were cleared.
def load_and_audit_resistant():
    bmc = pd.read_csv(BMCN_PATH, compression='gzip')
    REQUIRED = {
        'interaction_row', 'biosample', 'antibiotic', 'outer_fold',
        'observed_log2_mic', 'model_c_outer_test_predicted_log2_mic',
        'bmc_n_lower95_log2', 'bmc_n_upper95_log2'
    }
    missing = sorted(REQUIRED - set(bmc.columns))
    if missing:
        raise ValueError(f'NB41 prediction file lacks required columns: {missing}')
    
    if (len(bmc), bmc.biosample.nunique(), bmc.antibiotic.nunique()) != (50460, 9058, 26):
        raise ValueError('NB41 held-out table does not have 50,460 rows, 9,058 BioSamples, 26 antibiotics.')
    if bmc.interaction_row.duplicated().any() or bmc.duplicated(['biosample', 'antibiotic']).any():
        raise ValueError('Duplicate held-out observations detected.')
    if set(bmc.outer_fold) != set(range(1, 6)):
        raise ValueError('Expected five outer folds.')
    if (bmc.groupby('biosample').outer_fold.nunique() != 1).any():
        raise ValueError('A BioSample appears in more than one independent outer fold.')
    
    value_cols = [
        'observed_log2_mic', 'model_c_outer_test_predicted_log2_mic',
        'bmc_n_lower95_log2', 'bmc_n_upper95_log2'
    ]
    if not np.isfinite(bmc[value_cols].to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite MIC or interval values in NB41 predictions.')
    if not (bmc.bmc_n_upper95_log2 > bmc.bmc_n_lower95_log2).all():
        raise ValueError('Some BMC-N interval bounds are invalid.')
    
    bmc['antibiotic'] = bmc['antibiotic'].astype(str).str.lower().str.strip()
    bmc['absolute_model_c_error_log2'] = (
        bmc['observed_log2_mic'] - bmc['model_c_outer_test_predicted_log2_mic']
    ).abs()
    bmc['within_bmc_n_95pct'] = (
        (bmc['observed_log2_mic'] >= bmc['bmc_n_lower95_log2']) &
        (bmc['observed_log2_mic'] <= bmc['bmc_n_upper95_log2'])
    )
    
    # Existing approved observed-MIC resistance cutoffs for E. coli (mg/L).
    # These are strict > thresholds (not >=).
    
    # Use log2 thresholds directly; avoids numerical conversion and rounding.
    resistant = bmc.loc[
        ((bmc.antibiotic == 'cefepime') &
         (bmc.observed_log2_mic > np.log2(RESISTANCE_LIMIT_MG_L['cefepime']))) |
        ((bmc.antibiotic == 'gentamicin') &
         (bmc.observed_log2_mic > np.log2(RESISTANCE_LIMIT_MG_L['gentamicin'])))
    ].copy()
    
    counts = resistant.antibiotic.value_counts()
    for abx in RESISTANCE_LIMIT_MG_L:
        total = int((bmc.antibiotic == abx).sum())
        n_res = int(counts.get(abx, 0))
        print(f'{abx.title()}: {n_res:,} resistant observations of {total:,} held-out MIC observations')
        if n_res < 15:
            raise ValueError(f'Not enough observed-resistant pathogens for {abx}: {n_res}.')
    
    # The validated Model C dataset contains 449 cefepime and 416 gentamicin
    # observed-resistant E. coli under these chosen breakpoints.
    if (int(counts.get('cefepime', 0)), int(counts.get('gentamicin', 0))) != (449, 416):
        raise ValueError('Resistant cohort size differs from the audited Model C dataset (449, 416).')
    print('Confirmed: all candidate observations meet their antibiotic-specific resistance cutoff.')
    return bmc, resistant


In [ ]:
#@title Cell 52.3 — Load and audit the 50,460 independent BMC-N predictions
bmc, resistant = load_and_audit_resistant()


In [ ]:
#@title Cell 52.4 — Select 15 resistant pathogens per antibiotic without using coverage
# Rank absolute errors within EACH resistant antibiotic cohort, split into thirds,
# then choose five evenly spaced ranks per third (reproducible; no random seed).
# Selection does not use BMC-N interval endpoints or interval coverage.
# This cell also works if 52.3 was accidentally skipped (after running 52.2).
if 'resistant' not in globals() or 'bmc' not in globals():
    print('Cell 52.3 was not run; loading and auditing the NB41 input now.')
    bmc, resistant = load_and_audit_resistant()

def choose_five_evenly(df):
    if len(df) < 5:
        raise ValueError('Cannot select five pathogens from fewer than five candidates.')
    ranks = np.rint(np.linspace(0, len(df) - 1, 5)).astype(int)
    if len(set(ranks)) != 5:
        raise ValueError('Selection produced repeated ranks.')
    return df.iloc[ranks].copy()

selections = []
summary = []
for abx in ['cefepime', 'gentamicin']:
    pool = resistant.loc[resistant.antibiotic.eq(abx)].copy()
    pool = pool.sort_values(
        ['absolute_model_c_error_log2', 'biosample', 'outer_fold'],
        kind='mergesort'
    ).reset_index(drop=True)
    thirds = np.array_split(np.arange(len(pool)), 3)
    for label, indices in zip(['Small error', 'Moderate error', 'Large error'], thirds):
        group = choose_five_evenly(pool.iloc[indices])
        group['error_group'] = label
        group['rank_within_group'] = np.arange(1, 6)
        selections.append(group)

    cohort = pool
    summary.append({
        'antibiotic': abx,
        'resistance_cutoff_mg_L_exclusive': RESISTANCE_LIMIT_MG_L[abx],
        'n_all_heldout': int(bmc.antibiotic.eq(abx).sum()),
        'n_observed_resistant': len(cohort),
        'full_resistant_cohort_coverage_95pct': cohort.within_bmc_n_95pct.mean(),
        'full_resistant_cohort_mean_width_log2':
            (cohort.bmc_n_upper95_log2 - cohort.bmc_n_lower95_log2).mean(),
    })

selected = pd.concat(selections, ignore_index=True)
selected['observed_mic_mg_L'] = np.exp2(selected.observed_log2_mic)
selected['bmc_n_interval_width_log2'] = selected.bmc_n_upper95_log2 - selected.bmc_n_lower95_log2

for abx, bp in RESISTANCE_LIMIT_MG_L.items():
    sub = selected.loc[selected.antibiotic.eq(abx)]
    if len(sub) != 15 or sub.biosample.nunique() != 15:
        raise ValueError(f'Expected 15 distinct BioSamples for {abx}.')
    if not (sub.observed_log2_mic > np.log2(bp)).all():
        raise ValueError(f'Selection contains nonresistant MIC values for {abx}.')
    if sub.error_group.value_counts().to_dict() != {
        'Small error': 5, 'Moderate error': 5, 'Large error': 5
    }:
        raise ValueError(f'Wrong error-group composition for {abx}.')

summary_df = pd.DataFrame(summary)
summary_df['n_selected'] = 15
summary_df['selected_coverage_95pct_descriptive_only'] = [
    selected.loc[selected.antibiotic.eq(abx), 'within_bmc_n_95pct'].mean()
    for abx in summary_df.antibiotic
]
print('Resistant cohort and selected-case summary:')
display(summary_df.round(4))

selected_cols = [
    'antibiotic','biosample','interaction_row','outer_fold',
    'observed_log2_mic','observed_mic_mg_L',
    'model_c_outer_test_predicted_log2_mic',
    'bmc_n_lower95_log2','bmc_n_upper95_log2',
    'bmc_n_interval_width_log2','absolute_model_c_error_log2',
    'error_group','rank_within_group','within_bmc_n_95pct'
]
print('Selected 30 pathogen–antibiotic cases:')
display(selected[selected_cols].round(4))

In [ ]:
#@title Cell 52.5 — Draw publication-style black-and-white Figure 3
# Two vertically stacked panels share a common log2 MIC axis, making
# cefepime and gentamicin interval widths directly comparable.
# Circle = original Model C point prediction; x = observed MIC;
# horizontal segment = BMC-N 95% predictive MIC interval.
# Dashed vertical line marks the observed-MIC resistance threshold.

fig, axes = plt.subplots(2, 1, figsize=(6.7, 8.6), sharex=True)
all_x = np.concatenate([
    selected[c].to_numpy(dtype=float) for c in [
        'observed_log2_mic', 'model_c_outer_test_predicted_log2_mic',
        'bmc_n_lower95_log2', 'bmc_n_upper95_log2'
    ]
])
x_lo = float(np.floor(all_x.min() - 0.6))
x_hi = float(np.ceil(all_x.max() + 0.6))

for ax, (letter, abx) in zip(axes, [('A', 'cefepime'), ('B', 'gentamicin')]):
    sub = selected.loc[selected.antibiotic.eq(abx)].copy()
    # Keep three 5-row error groups visibly ordered, top to bottom.
    order = {'Small error': 0, 'Moderate error': 1, 'Large error': 2}
    sub['group_order'] = sub.error_group.map(order)
    sub = sub.sort_values(['group_order', 'rank_within_group']).reset_index(drop=True)
    yy = 15 - np.arange(15)

    for i, row in sub.iterrows():
        x1 = row['bmc_n_lower95_log2']
        x2 = row['bmc_n_upper95_log2']
        ax.hlines(yy[i], x1, x2, color='0.53', linewidth=2.0, zorder=1)
        ax.vlines([x1, x2], yy[i] - 0.15, yy[i] + 0.15,
                  color='0.53', linewidth=0.7, zorder=1)

    ax.scatter(sub.model_c_outer_test_predicted_log2_mic, yy,
               marker='o', s=26, facecolor='white', edgecolor='black',
               linewidth=0.85, zorder=3, label='Model C predicted MIC')
    ax.scatter(sub.observed_log2_mic, yy,
               marker='x', s=32, color='black', linewidth=1.0,
               zorder=4, label='Observed MIC')

    threshold_log2 = np.log2(RESISTANCE_LIMIT_MG_L[abx])
    ax.axvline(threshold_log2, color='0.35', linestyle='--',
               linewidth=0.75, zorder=0)
    ax.axhline(10.5, color='0.80', linewidth=0.65)
    ax.axhline(5.5, color='0.80', linewidth=0.65)
    ax.grid(axis='x', linewidth=0.4, color='0.91', zorder=0)
    ax.set_yticks(yy)
    ax.set_yticklabels(sub.biosample, fontsize=7.2)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', labelsize=9)
    ax.set_ylim(0.4, 15.9)
    ax.set_xlim(x_lo, x_hi)
    ax.set_title(
        f'{letter}  {abx.capitalize()} — observed MIC > '
        f'{RESISTANCE_LIMIT_MG_L[abx]:g} mg/L',
        loc='left', fontsize=10, pad=10
    )
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

axes[1].set_xlabel('log$_2$ MIC (mg/L)', fontsize=10)
axes[0].tick_params(axis='x', labelbottom=True)
axes[0].set_xlabel('log$_2$ MIC (mg/L)', fontsize=10)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='0.53', lw=2, label='BMC-N 95% interval'),
    Line2D([0], [0], marker='o', color='black', markerfacecolor='white',
           linestyle='None', markersize=5.4, label='Model C prediction'),
    Line2D([0], [0], marker='x', color='black', linestyle='None',
           markersize=6, label='Observed MIC'),
    Line2D([0], [0], color='0.35', linestyle='--', lw=0.8,
           label='R threshold')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.52, 0.01))
fig.subplots_adjust(left=0.24, right=0.98, top=0.96, bottom=0.12, hspace=0.30)

FIG_PNG = OUT / '52_Figure3_BMCN_Cefepime_Gentamicin_Resistant.png'
FIG_PDF = OUT / '52_Figure3_BMCN_Cefepime_Gentamicin_Resistant.pdf'
fig.savefig(FIG_PNG, dpi=350, bbox_inches='tight')
fig.savefig(FIG_PDF, bbox_inches='tight')
plt.show()
print('Saved figure:', FIG_PNG)
print('Saved PDF:', FIG_PDF)

In [ ]:
#@title Cell 52.6 — Export selected pathogen tables and reproducibility audit
import json

SELECTED_CSV = OUT / '52_Figure3_Selected_30_Resistant_Pathogens.csv'
SUMMARY_CSV = OUT / '52_Figure3_Resistance_And_Coverage_Audit.csv'
selected[selected_cols].to_csv(SELECTED_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)

audit = {
    'notebook': '52_BMC_Figure3_Resistant_Pathogens_v0.3',
    'source_notebook': 41,
    'source_file': str(BMCN_PATH),
    'source_file_sha256': sha256_file(BMCN_PATH),
    'resistance_cutoffs_mg_L_exclusive': RESISTANCE_LIMIT_MG_L,
    'classification_uses': 'observed MIC, never predicted MIC',
    'selection_basis': 'absolute original Model C error among resistant held-out observations',
    'selection_rule': '5 evenly spaced ranks from each of three equal-sized sorted error bands per antibiotic',
    'selected_cases': int(len(selected)),
    'selected_per_antibiotic': 15,
    'number_of_outer_test_observations': 50460,
    'selection_does_not_use_interval_coverage': True,
    'notes': 'Selected-case coverage is descriptive, not an unbiased estimate of population coverage.'
}
AUDIT_PATH = OUT / '52_Figure3_Selection_Provenance.json'
AUDIT_PATH.write_text(json.dumps(audit, indent=2), encoding='utf-8')
print('Saved:', SELECTED_CSV)
print('Saved:', SUMMARY_CSV)
print('Saved:', AUDIT_PATH)
print('NB52 complete. Figure 3 uses unchanged NB41 Bayesian intervals and Model C predictions.')